In [ ]:
import uuid

RUN_ID = str(uuid.uuid4())
RUN_ID_SHORT = RUN_ID[:5]
print(f"run id: {RUN_ID} (short: {RUN_ID_SHORT})")

# Initial Cleaning

This section handles cleaning the raw data from the review and metadata files and joining them together.
I then export this and comment everything out and use the newly exported dataset as the original source of truth.

### Importing

In [75]:
import pandas as pd
import gzip
import json

def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield json.loads(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
    if i > 750000: 
      break
  return pd.DataFrame.from_dict(df, orient='index')

df_reviews = getDF('data/Books_5.json.gz')
df_metadata = getDF('data/meta_Books.json.gz')

### Renaming

In [76]:
review_column_rename_map = {
    "overall": "user_rating",
    "verified": "user_verified_purchase",
    "reviewTime": "user_review_date_raw",
    "reviewerID": "user_id",
    "asin": "book_id",
    "style": "book_format",
    "reviewerName": "user_name",
    "reviewText": "user_review_text",
    "summary": "user_review_summary",
    "unixReviewTime": "user_review_timestamp",
    "vote": "user_review_helpful_votes",
}

metadata_column_rename_map = {
    "category": "book_category",
    "description": "book_description",
    "title": "book_title",
    "brand": "book_brand",
    "rank": "book_rank",
    "price": "book_price",
    "asin": "book_id",
    "imageURL": "book_image_url",
}

df_reviews = df_reviews.rename(columns=review_column_rename_map)
df_metadata = df_metadata.rename(columns=metadata_column_rename_map)

### Dropping rows and columns

In [77]:

review_columns_to_drop = [
    "user_review_date_raw",
    "user_verified_purchase",
    "user_name",
    "user_review_helpful_votes",
    "image"
]

metadata_columns_to_drop = [
    "tech1", 
    "fit", 
    "tech2", 
    "feature",  
    "also_buy", 
    "main_cat", 
    "similar_item", 
    "date",
    'imageURLHighRes',
    'also_view'
]

df_reviews.drop(columns=review_columns_to_drop, inplace=True)
df_metadata.drop(columns=metadata_columns_to_drop, inplace=True)

### Cleaning

In [78]:
from datetime import datetime
import re

def clean_book_format(text):
    """
        Input: {'Format:': ' Paperback'} 
        Output: 'paperback'
    """
    if isinstance(text, dict):
        value = str(list(text.values())[0]).strip().lower()
        return value
    return None

def convert_unix_timestamp_to_year(timestamp):
    """
        Input: 1112140800 (March 30 2005)
        Output: 2005
    """
    year = datetime.fromtimestamp(timestamp).year
    return year

def parse_rank_from(book_rank):
    """
        Input: '1,349,781 in Books ('
        Output: 1349781
    """
    if book_rank is None or isinstance(book_rank, list):
        book_rank = "" if book_rank is None else " ".join(map(str, book_rank))
    match = re.search(r'\d[\d,]*', str(book_rank))
    if not match:
        return None
    return int(match.group().replace(',', ''))

def remove_dollar_sign(text):
    return text.strip("$")

df_reviews["book_format"] = df_reviews["book_format"].apply(clean_book_format)
df_reviews["user_review_timestamp"] = df_reviews["user_review_timestamp"].apply(convert_unix_timestamp_to_year)
df_metadata["book_price"] = df_metadata["book_price"].apply(remove_dollar_sign)
df_metadata["book_rank"] = df_metadata["book_rank"].apply(parse_rank_from)

### Preprocessing

In [79]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
import torch

#
# Fillna with median
# 
df_metadata["book_rank"] = df_metadata["book_rank"].fillna(df_metadata["book_rank"].median())

#
# Scaling the book rank
#
df_metadata["book_rank_scaled"] = StandardScaler().fit_transform(df_metadata[["book_rank"]])
book_rank_scaled_idx = torch.tensor(df_metadata["book_rank_scaled"].values, dtype=torch.float32)

#
# Using sentence transformer to create text embeddings for book titles
#
sentence_transformer_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2", # 384 dim embeddings
    device="cpu",
)

embeddings = sentence_transformer_model.encode(
    df_metadata["book_title"].astype(str).tolist(),
    batch_size=256,
    convert_to_numpy=True,
    show_progress_bar=True,
)

book_title_emb = torch.from_numpy(embeddings).float()
book_title_emb_dim = book_title_emb.shape[1]

Batches: 100%|██████████| 2930/2930 [12:52<00:00,  3.79it/s]


In [80]:
import numpy as np

user_ids = df_reviews['user_id'].unique()
book_ids = df_metadata['book_id'].unique()

user2idx = {u: i for i, u in enumerate(user_ids)}
book2idx = {b: i for i, b in enumerate(book_ids)}

df_reviews["user_idx"] = df_reviews["user_id"].map(user2idx)
df_reviews["book_idx"] = df_reviews["book_id"].map(book2idx)

df_reviews = df_reviews.dropna(subset=["book_idx"])
df_reviews["book_idx"] = df_reviews["book_idx"].astype(int)

df_metadata["book_idx"] = df_metadata["book_id"].map(book2idx)
df_metadata = df_metadata.dropna(subset=["book_idx"]).set_index("book_idx").sort_index()

n_users, n_books = len(user2idx), len(book2idx)

user_pos_books = df_reviews.groupby("user_idx")["book_idx"].apply(set).to_dict()
all_book_idxs = np.arange(n_books)

In [81]:
df_reviews.columns

Index(['user_rating', 'user_id', 'book_id', 'book_format', 'user_review_text',
       'user_review_summary', 'user_review_timestamp', 'user_idx', 'book_idx'],
      dtype='str')

In [82]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class TwoTowersDataset(Dataset):
    def __init__(self, df_reviews):
        self.users = df_reviews["user_idx"].values
        self.books = df_reviews["book_idx"].values

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        book_idx = self.books[idx]
        return {
            "user_idx": torch.tensor(self.users[idx], dtype=torch.long),
            "book_idx": torch.tensor(book_idx, dtype=torch.long),
            "book_rank_scaled": book_rank_scaled_idx[book_idx],
            "book_title_emb": book_title_emb[book_idx]
        }


# Training

In [83]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# # # # # # # # # # # # # # # # # #
#
# USER TOWER
#
#   - user_idx
#
# # # # # # # # # # # # # # # # # #


class UserTower(nn.Module):
    def __init__(self, n_users, embedding_dim, user_embedding_dim=128):
        super().__init__()

        self.user_idx_embedding = nn.Embedding(n_users, user_embedding_dim)

        self.user_mlp = nn.Sequential(
            nn.Linear(user_embedding_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, embedding_dim),
        )

    def forward(self, user_idx):
        user_emb = self.user_idx_embedding(user_idx)
        return self.user_mlp(user_emb)


# # # # # # # # # # # # # # # # # #
#
# ITEM TOWER
#
#   - book_idx
#   - book_rank_scaled
#
# # # # # # # # # # # # # # # # # #


class ItemTower(nn.Module):
    def __init__(self, n_books, embedding_dim, book_title_emb_dim, book_embedding_dim=128):
        super().__init__()

        self.book_idx_embedding = nn.Embedding(n_books, book_embedding_dim)

        self.item_mlp = nn.Sequential(
            nn.Linear(book_embedding_dim + 1 + book_title_emb_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, embedding_dim),
        )

    def forward(self, book_idx, book_rank_scaled, book_title_emb):
        book_idx_emb = self.book_idx_embedding(book_idx)

        x = torch.cat([
            book_idx_emb, 
            book_rank_scaled.unsqueeze(1),
            book_title_emb
        ], dim=1)

        return self.item_mlp(x)


# # # # # # # # # # # # # # # # # #
#
# TWO TOWERS
#
# # # # # # # # # # # # # # # # # #


class TwoTowers(nn.Module):
    def __init__(self, user_tower: UserTower, item_tower: ItemTower):
        super().__init__()
        self.user_tower = user_tower
        self.item_tower = item_tower

    def forward(self, 
                user_idx, 
                book_idx, 
                book_rank_scaled,
                book_title_emb
                ):
        user_emb = self.user_tower(user_idx)
        item_emb = self.item_tower(book_idx, book_rank_scaled, book_title_emb)

        user_emb = F.normalize(user_emb, p=2, dim=1)
        item_emb = F.normalize(item_emb, p=2, dim=1)

        return user_emb, item_emb


### Train/test split

Since the dataset is going to be extremely sparse. I am using a user-level split rather than a random split. This is because a random split would more likely use user IDs that were not seen at all during training, whose embeddings are completely random as well. 

This ensures that users in the validation set have learned embeddings, but use examples that weren't memorized

In [84]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embedding_dim = 64
n_epochs = 50
lr = 1e-3
temperature = 0.1  

In [85]:
# 1. Enforce min-interaction filter: drop users with < 5 reviews entirely.
MIN_INTERACTIONS = 4
counts = df_reviews["user_idx"].value_counts()
keep_users = counts[counts >= MIN_INTERACTIONS].index
df_reviews = df_reviews[df_reviews["user_idx"].isin(keep_users)].copy()

# 2. Now leave-one-out split (every remaining user has >=5, so all are eligible).
val_idx = df_reviews.groupby("user_idx").sample(n=1, random_state=42).index
df_val = df_reviews.loc[val_idx]
df_train = df_reviews.drop(index=val_idx)

print(f"users kept: {df_reviews['user_idx'].nunique()}, "
      f"train rows: {len(df_train)}, val rows: {len(df_val)}")

# logQ correction: in-batch negatives are sampled proportional to how often a
# book appears in the data, so popular books are unfairly over-penalized. We
# subtract log(sampling_prob) from each item's logit to correct for that. This
# is what stops one popular title from topping everyone's list.
book_counts = np.ones(n_books)  # +1 smoothing so unseen books aren't -inf
for b in df_train["book_idx"].values:
    book_counts[b] += 1
book_log_prob = torch.tensor(
    np.log(book_counts / book_counts.sum()), dtype=torch.float32, device=device
)

train_dataset = TwoTowersDataset(df_train)
val_dataset = TwoTowersDataset(df_val)

train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True, drop_last=True)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)


users kept: 27323, train rows: 170188, val rows: 27323


## Saving / Loading state

In [ ]:
import pickle

def save_progress(model: TwoTowers, optimizer, epoch: int):
    #
    # Architecture
    #
    torch.save({
        "run_id": RUN_ID,
        "n_users": n_users,
        "n_books": n_books,
        "embedding_dim": embedding_dim,
        "book_title_emb_dim": book_title_emb_dim,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "epoch": epoch,
        "book_title_emb": book_title_emb.cpu(),
        "book_rank_scaled_idx": book_rank_scaled_idx.cpu(),
        "book_log_prob": book_log_prob.cpu(),
    }, f"checkpoint_{RUN_ID_SHORT}_epoch{epoch}.pth")

    #
    # Artifacts
    #
    with open(f"artifacts_{RUN_ID_SHORT}_epoch{epoch}.pkl", "wb") as f:
        pickle.dump({
            "user2idx": user2idx,
            "book2idx": book2idx,
            "user_pos_books": user_pos_books,
        }, f)

    #
    # Book index to know what prediction was made
    #
    df_metadata.to_pickle(f"df_metadata_{RUN_ID_SHORT}_epoch{epoch}.pkl")

    print(
        f"saved checkpoint_{RUN_ID_SHORT}_epoch{epoch}.pth, "
        f"artifacts_{RUN_ID_SHORT}_epoch{epoch}.pkl, "
        f"df_metadata_{RUN_ID_SHORT}_epoch{epoch}.pkl"
    )

## Loading 

In [ ]:
# # Reload everything from disk to pick up where we left off.
# # Run this INSTEAD of the fresh-training init cell above. Assumes the class
# # defs (UserTower, ItemTower, TwoTowers) above have already been executed.
# import pickle
# import torch
# import pandas as pd

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ckpt = torch.load("checkpoint_epoch10.pth", map_location=device)

# # Rebuild the model with the exact shapes it was trained with, then load weights.
# user_tower = UserTower(ckpt["n_users"], ckpt["embedding_dim"]).to(device)
# item_tower = ItemTower(
#     ckpt["n_books"], ckpt["embedding_dim"], ckpt["book_title_emb_dim"]
# ).to(device)
# model = TwoTowers(user_tower, item_tower).to(device)
# model.load_state_dict(ckpt["model_state"])

# # Precomputed item features (indexed by book_idx) — restored, not recomputed.
# book_title_emb = ckpt["book_title_emb"].to(device)
# book_rank_scaled_idx = ckpt["book_rank_scaled_idx"]
# book_log_prob = ckpt["book_log_prob"].to(device)

# # ID mappings + inference support data.
# with open("artifacts_10.pkl", "rb") as f:
#     artifacts = pickle.load(f)
# user2idx = artifacts["user2idx"]
# book2idx = artifacts["book2idx"]
# user_pos_books = artifacts["user_pos_books"]

# df_metadata = pd.read_pickle("df_metadata_10.pkl")

# n_users = ckpt["n_users"]
# n_books = ckpt["n_books"]
# embedding_dim = ckpt["embedding_dim"]
# book_title_emb_dim = ckpt["book_title_emb_dim"]

# # Create the optimizer LAST, after `model` is final, then load its state.
# # Do NOT rebind `model` after this point.
# optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
# optimizer.load_state_dict(ckpt["optimizer_state"])
# start_epoch = ckpt["epoch"]

# print(f"loaded checkpoint - epoch {ckpt['epoch']}, "
#       f"n_users={n_users}, n_books={n_books}")

In [ ]:
# Fresh-training init. Run this (in a clean kernel) right before the training loop.
# IMPORTANT: build `model` FIRST and create `optimizer` LAST. Never rebind `model`
# after the optimizer is created, or the optimizer will update an orphaned model
# while the one you train/evaluate never changes (flat loss, val_recall stuck at 0).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

user_tower = UserTower(n_users, embedding_dim).to(device)
item_tower = ItemTower(n_books, embedding_dim, book_title_emb_dim).to(device)
model = TwoTowers(user_tower, item_tower).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
start_epoch = 0

# sanity check: the optimizer must own the exact tensors we train
assert next(iter(optimizer.param_groups[0]["params"])) is next(model.parameters())
print(f"initialized fresh model - n_users={n_users}, n_books={n_books}")

In [88]:
@torch.no_grad()
def recall_at_k(model, df_eval, k=10, max_users=2000):
    """Fraction of held-out positives that land in the user's top-k. This is the
    metric that actually reflects recommendation quality (the old sims>0 accuracy
    was near 100% even while the model was collapsing)."""
    model.eval()

    all_books = torch.arange(n_books, device=device)
    all_ranks = book_rank_scaled_idx.to(device)
    book_vectors = F.normalize(model.item_tower(all_books, all_ranks, book_title_emb), p=2, dim=1)

    sample = df_eval.sample(n=min(max_users, len(df_eval)), random_state=0)
    hits = 0
    for user_idx, book_idx in zip(sample["user_idx"].values, sample["book_idx"].values):
        user_tensor = torch.tensor([user_idx], device=device)
        user_vector = F.normalize(model.user_tower(user_tensor), p=2, dim=1)
        scores = (user_vector @ book_vectors.T).squeeze(0)

        # hide books this user already reviewed, except the held-out target
        seen = user_pos_books.get(user_idx, set()) - {book_idx}
        if seen:
            scores[list(seen)] = float("-inf")

        topk = torch.topk(scores, k).indices
        hits += int(book_idx in topk.cpu().numpy())
    return hits / len(sample)


for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for batch in train_dataloader:
        batch_user_idx = batch["user_idx"].to(device)
        batch_book_idx = batch["book_idx"].to(device)
        batch_book_rank_scaled = batch["book_rank_scaled"].to(device)
        batch_book_title_emb = batch["book_title_emb"].to(device)

        user_vec, item_vec = model(
            batch_user_idx, 
            batch_book_idx, 
            batch_book_rank_scaled,
            batch_book_title_emb
        )  # L2-normalized
        
        # (B, B): row i = user i, col j = the positive item of pair j.
        # The diagonal is the true (user, item) pair; every off-diagonal item is
        # an in-batch negative for that user.
        logits = (user_vec @ item_vec.T) / temperature
        logits = logits - book_log_prob[batch_book_idx].unsqueeze(0)   # logQ correction

        # Accidental-hit masking: a popular book can appear in several rows of
        # the batch, so an off-diagonal column may actually be another row's
        # true positive. Mask those so they aren't scored as negatives.
        same_item = batch_book_idx.unsqueeze(0) == batch_book_idx.unsqueeze(1)
        same_item.fill_diagonal_(False)
        logits = logits.masked_fill(same_item, float("-inf"))

        labels = torch.arange(user_vec.size(0), device=device)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_dataloader)
    val_recall = recall_at_k(model, df_val, k=10)
    print(
        f"epoch {epoch + 1}/{n_epochs} - "
        f"train_loss: {train_loss:.4f} - val_recall@10: {val_recall:.4f}"
    )

    if epoch % 10 == 0:
        save_progress(model, optimizer, epoch)


epoch 1/50 - train_loss: 7.9302 - val_recall@10: 0.0000
saved checkpoint_epoch0.pth, artifacts.pkl, df_metadata.pkl
epoch 2/50 - train_loss: 7.9298 - val_recall@10: 0.0000
epoch 3/50 - train_loss: 7.9287 - val_recall@10: 0.0000
epoch 4/50 - train_loss: 7.9277 - val_recall@10: 0.0000
epoch 5/50 - train_loss: 7.9314 - val_recall@10: 0.0000
epoch 6/50 - train_loss: 7.9290 - val_recall@10: 0.0000
epoch 7/50 - train_loss: 7.9329 - val_recall@10: 0.0000
epoch 8/50 - train_loss: 7.9275 - val_recall@10: 0.0000
epoch 9/50 - train_loss: 7.9355 - val_recall@10: 0.0000
epoch 10/50 - train_loss: 7.9314 - val_recall@10: 0.0000
epoch 11/50 - train_loss: 7.9288 - val_recall@10: 0.0000
saved checkpoint_epoch10.pth, artifacts.pkl, df_metadata.pkl
epoch 12/50 - train_loss: 7.9350 - val_recall@10: 0.0000
epoch 13/50 - train_loss: 7.9350 - val_recall@10: 0.0000


KeyboardInterrupt: 

In [ ]:
# Collapse check: if user embeddings had collapsed to one direction (the old
# "Golden Fool for everyone" bug), this mean cosine would sit near ~0.99. After
# in-batch softmax training it should be substantially lower.
with torch.no_grad():
    u = F.normalize(model.user_tower(torch.arange(n_users, device=device)), p=2, dim=1)
    sample = u[torch.randperm(n_users)[:1000]]
    print("mean pairwise user cosine:", (sample @ sample.T).mean().item())


In [ ]:
@torch.no_grad()
def recommend_for_user(user_idx, k=100, exclude_seen=True):
    model.eval()

    # Embed every book once
    all_books = torch.arange(n_books, device=device)
    all_ranks = book_rank_scaled_idx.to(device)
    book_vectors = F.normalize(model.item_tower(all_books, all_ranks, book_title_emb), p=2, dim=1)

    # Embed the single user
    user_tensor = torch.tensor([user_idx], device=device)
    user_vector = F.normalize(model.user_tower(user_tensor), p=2, dim=1)

    # Cosine similarity of this user against every book
    scores = (user_vector @ book_vectors.T).squeeze(0)

    # Hide books the user has already reviewed
    if exclude_seen:
        seen = list(user_pos_books.get(user_idx, set()))
        scores[seen] = float("-inf")

    top_scores, top_idx = torch.topk(scores, k)
    return top_idx.cpu().numpy(), top_scores.cpu().numpy()


user_idx = 1000
top_book_idxs, top_scores = recommend_for_user(user_idx, k=100)

recommendations = df_metadata.loc[top_book_idxs, ["book_id", "book_title"]].copy()
recommendations["score"] = top_scores
recommendations.reset_index(drop=True)


In [ ]:
reviewed_book_ids = df_reviews[df_reviews["user_idx"] == 1000]["book_idx"].values
df_metadata.iloc[reviewed_book_ids]